Importing libraries

In [32]:
import os
import gdown
import pandas as pd

Definindo variáveis p/ funções: 
- file_id: google drive file id
- output: caminho de saída do arquivo baixado

In [33]:
file_id = '1Ki8nvHJx91LM_vTZ7motgNhCF9fjyfGg'
output = 'data/raw/dados_projeto.csv'

Função para baixar o arquivo CSV do Google Drive

In [34]:
def download_csv_from_drive(file_id: str, output_path: str) -> None:
    """
    Baixa um arquivo CSV do Google Drive usando gdown.

    Args:
        file_id (str): ID do arquivo no Google Drive.
        output_path (str): Caminho onde salvar o arquivo baixado.
    """
    # Cria a pasta se não existir
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Sempre baixa o arquivo, sobrescrevendo se necessário
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, output_path, quiet=False)
    print(f"Arquivo baixado para {output_path} com sucesso.")

Função pra ler e tratar o arquivo base

In [35]:
def load_and_clean_data(filepath: str) -> pd.DataFrame:
    """
    Carrega um arquivo CSV e realiza limpeza básica dos dados.

    Args:
        filepath (str): Caminho para o arquivo CSV.

    Returns:
        pd.DataFrame: DataFrame limpo.
    """
    df = pd.read_csv(filepath)
    
    # Exemplo: remover nulos, converter tipos de dados, etc.
    df['generic_model_name'] = df['model_name'].str.split().str[0].str.split('-').str[0].str.upper() #extrai a parte genérica do nome do modelo (ex: "gpt-3.5-turbo" -> "gpt")
    df['clean_model_name'] = df['model_name'].str.split('(').str[0]
    df['reasoning level'] = df['generic_model_name'].str.extract(r'\((.*?)\)', expand=False).fillna('Unknown')
    #transformar a coluna de ano de lançamento em inteiro, forçando erros a se tornarem NaN
    df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce').astype('Int64')
    # Converter tipos de dados para os mais apropriados automaticamente
    df = df.convert_dtypes()
    # Remover linhas com valores nulos em colunas críticas
    df = df.dropna(how='all') #incluir subset=['coluna1', 'coluna2'] para especificar colunas que não podem ser nulas

    return df

In [36]:
def standardize_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza campos do DataFrame para uso em plotagem ou outras funções.

    Args:
        df (pd.DataFrame): DataFrame a ser padronizado.

    Returns:
        pd.DataFrame: DataFrame com campos padronizados.
    """
    # Exemplo: converter tipos, renomear colunas, etc.
    # Ajuste conforme os dados específicos
    df = df.copy()
    # Converter colunas numéricas
    numeric_cols = df.select_dtypes(include=['str']).columns
    for col in numeric_cols:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except:
            pass
    # Padronizar nomes de colunas para lowercase
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df

In [37]:
def save_dataframe(df: pd.DataFrame, output_path: str) -> None:
    """
    Salva um DataFrame em um arquivo CSV.

    Args:
        df (pd.DataFrame): DataFrame a ser salvo.
        output_path (str): Caminho onde salvar o arquivo CSV.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    try:
        df.to_csv(output_path, index=False)
        print(f"DataFrame salvo em {output_path} com sucesso.")
    except Exception as e:
        print(f"Erro ao salvar o DataFrame em {output_path}: {e}")
    

Aplicando as funções em ordem (conforme o orquestrador na main.py estaria chamando)

In [38]:
download_csv_from_drive(file_id, output)

Downloading...
From: https://drive.google.com/uc?id=1Ki8nvHJx91LM_vTZ7motgNhCF9fjyfGg
To: f:\Programações\Faculdade\Ciências dos Dados\Trabalhos\CienciaDosDados-Trabalho1\notebooks\data\raw\dados_projeto.csv
100%|██████████| 99.9k/99.9k [00:00<00:00, 709kB/s]

Arquivo baixado para data/raw/dados_projeto.csv com sucesso.


In [39]:
df = load_and_clean_data(output)

In [40]:
df = standardize_fields(df)

In [41]:
save_dataframe(df, 'data/processed/dados_projeto_limpos.csv')

DataFrame salvo em data/processed/dados_projeto_limpos.csv com sucesso.
